# Day 54: Measure Cold Start Latency

**Layer:** Production


## Concept Overview

Cold start latency is the time from container launch to first successful request. For LLMs, the dominant components are: container pull, model weight loading from disk, and GPU memory allocation.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Cold Start Components

Cold start latency is the time from container launch to first successful request. For LLMs, the domi...


In [ ]:
import time
import numpy as np

def model_cold_start(params_b, disk_bw_gb_s=3.0, gpu_bw_gb_s=20.0):
    model_gb = params_b * 2  # FP16
    t_disk = model_gb / disk_bw_gb_s
    t_gpu = model_gb / gpu_bw_gb_s
    t_alloc = 2.0  # KV cache allocation
    t_warmup = 3.0  # JIT compilation
    return {'disk': t_disk, 'gpu_xfer': t_gpu, 'alloc': t_alloc,
            'warmup': t_warmup, 'total': t_disk+t_gpu+t_alloc+t_warmup}

print("Cold start breakdown by model size:")
print(f"{'Model':<20} {'Disk (s)':>10} {'GPU (s)':>8} {'Total (s)':>10}")
for params, name in [(1,'1B'),(8,'8B'),(70,'70B'),(405,'405B')]:
    t = model_cold_start(params)
    print(f"{name+' model':<20} {t['disk']:>10.1f} {t['gpu_xfer']:>8.1f} {t['total']:>10.1f}")
print()
print("Mitigation strategies:")
for s,d in [('Model warm pool','Keep N idle replicas loaded'),
             ('Checkpoint sharding','Load layers in parallel from multiple disks'),
             ('Predictive scaling','Scale up 5 min before expected peak')]:
    print(f"  {s:<25} {d}")

## Experiments: Try These

1. Extend the implementation with an additional feature.
2. Benchmark on real hardware and compare to theoretical estimates.
3. Connect this to a previous day's implementation.


## Key Takeaways

- Cold start latency is the time from container launch to first successful request.
- For a 70B model (140GB FP16), disk loading at 3 GB/s takes 47 seconds — this is why warm pools are not optional for latency-sensitive production deployments..
- Day 54 implementation complete.
